In [1]:
import os
import re 
import json
import random
import numpy as np 
import pandas as pd
from pathlib import Path
from PIL import Image, UnidentifiedImageError
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [ ]:
INPUT_DIR = Path('/kaggle/input/datasets/mayukh18/fashion200k-dataset')
OUTPUT_DIR = Path('/kaggle/working/processed')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
MIN_PX = 50
NUM_USERS = 1000
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print('INPUT_DIR :', INPUT_DIR)
print('OUTPUT_DIR :', OUTPUT_DIR)
print('Exists :',INPUT_DIR.exists())

In [ ]:
def read_labels(labels_dir: Path) -> pd.DataFrame:
    records = []
    label_files = list(labels_dir.glob('*_detect_all.txt')) #Doc file *_detect_all.txt trong labels
    
    print(f'Tìm thấy {len(label_files)} file labels:{[f.name for f in label_files]}')

    for lf in label_files:
        split = 'train' if 'train' in lf.name else 'test'
        with open(lf, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) < 3:
                    continue
                img_rel = parts[0]           #Đường dẫn tương đối
                det_score = float(parts[1])   #Điểm
                desc = ' '.join(parts[2:])   #Mô tả sản phẩm
                records.append({
                    'img_rel': img_rel,
                    'det_score': det_score,
                    'description': desc,
                    'split_orig': split,
                })
    df = pd.DataFrame(records)
    print(f'Tổng dòng labels: {len(df)}')
    return df
    
labels_dir = INPUT_DIR / 'labels' / 'labels'
df_labels = read_labels(labels_dir)
df_labels.head(3)

In [ ]:
#Trích xuất metadata từ đường dẫn
def parse_path(img_rel: str) -> dict:
    parts = Path(img_rel).parts
    try:
        return {
            'gender': parts[0], #women
            'category': parts[1], #dresses
            'subcategory': parts[2], #casual_and_day_dresses
            'product_id': parts[3], #54777928
            'filename': parts[4], #54777928_0.jpeg
        }
    except IndexError:
        return {'gender':'','category':'','subcategory':'','product_id':'','filename':''}

parsed = df_labels['img_rel'].apply(parse_path)
df_labels = pd.concat([df_labels, pd.DataFrame(list(parsed))], axis=1)

#Mỗi sản phẩm có nhiều góc chụp -> chỉ lấy ảnh đầu tiên làm đại diện
mask_front = df_labels['filename'].str.contains(r'_0\.', regex=True)
df_front = df_labels[mask_front].copy()
print(f'Tổng dòng ban đầu: {len(df_labels)}')
print(f'Sau khi lọc: {len(df_front)}')

#Loại trùng product_id (Trường hợp 1 product có 2 dòng _0)
df_front = df_front.drop_duplicates(subset='product_id', keep='first')
print(f'Sau khi bỏ trùng: {len(df_front)}')

In [ ]:
#Kiểm tra file ảnh tồn tại + không bị corrupt
def is_valid_image(rel_path: str, root: Path, min_px:int = MIN_PX) -> bool:
    full = root/rel_path
    if not full.exists():
        return False
    try:
        with Image.open(full) as img:
            w, h = img.size
            if w < min_px or h < min_px:
                return False
            img.verify()
        return True
    except Exception:
        return False

valid_mask = df_front['img_rel'].apply(lambda p: is_valid_image(p, INPUT_DIR))
df_valid = df_front[valid_mask].copy().reset_index(drop=True)

print(f'Ảnh hợp lệ: {valid_mask.sum()}')
print(f'Ảnh bị loại: {(~valid_mask).sum}') #Không tồn tại, corrupt, quá nhỏ

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s,.'\-]",' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_valid['description'] = df_valid['description'].apply(clean_text)

df_valid = df_valid[df_valid['description'].str.split().str.len() >=3]
print(f'Sau khi lọc text ngắn: {len(df_valid)} sản phẩm')

df_valid[['product_id','category','description']].head(10)

In [ ]:
img_out_dir = OUTPUT_DIR / 'images'
img_out_dir.mkdir(exist_ok=True)

def resize_image(rel_path: str, src_root: Path, dst_dir: Path, size: int = IMG_SIZE) -> str:
    src = src_root / rel_path
    dst = dst_dir / Path(rel_path).name #Giữ tên file gốc
    if dst.exists():
        return str(dst)
    with Image.open(src) as img:
        img = img.convert('RGB')
        img = img.resize((size,size), Image.LANCZOS)
        img.save(dst, 'JPEG', quality=90)
    return str(dst)
print(f'Resize {len(df_valid)} ảnh về {IMG_SIZE}x{IMG_SIZE}...')
output_paths = []
for i, row in df_valid.iterrows():
    path = resize_image(row['img_rel'], INPUT_DIR, img_out_dir)
    output_paths.append(path)
    if (len(output_paths)) % 5000 == 0:
        print(f' {len(output_paths)}/{len(df_valid)}')

df_valid['image_path'] = output_paths
print('RESIZE COMPLETE!')
        

In [ ]:
df_meta = df_valid.reset_index(drop=True)
df_meta.insert(0, 'item_id', df_meta.index) #item_id = 0,1,2,..

#Giữ lại các cột cần thiết
df_meta = df_meta[['item_id','product_id','description','category','subcategory','gender','image_path']]
#Dùng subcategory làm name
df_meta.insert(2,'name', df_meta['subcategory'].str.replace('_',' '))

out_meta = OUTPUT_DIR / 'metadata.csv'
df_meta.to_csv(out_meta, index=False)
print(f'metadata.csv: {len(df_meta)} dòng -> {out_meta}')
df_meta.head(3)

In [ ]:
item_ids = df_meta['item_id'].tolist()
rows = []
for user_id in range(NUM_USERS):
    n = random.randint(10, 50)
    sampled = random.sample(item_ids, min(n, len(item_ids)))
    for item_id in sampled:
        # rating lệch về cao — giống hành vi thực tế
        rating = random.choices([1,2,3,4,5], weights=[5,10,20,35,30])[0]
        rows.append({'user_id': user_id, 'item_id': item_id, 'rating': rating})

df_inter = pd.DataFrame(rows)
out_inter = OUTPUT_DIR / 'interactions.csv'
df_inter.to_csv(out_inter, index=False)
print(f'interactions.csv: {len(df_inter)} dòng, {NUM_USERS} users -> {out_inter}')

In [ ]:
train_idx, temp_idx = train_test_split(
    df_meta.index.tolist(), test_size=0.2,
    stratify=df_meta['category'], random_state=RANDOM_SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=0.5,
    stratify=df_meta.loc[temp_idx, 'category'], random_state=RANDOM_SEED
)

splits = {'train': train_idx, 'val': val_idx, 'test': test_idx}
with open(OUTPUT_DIR / 'splits.json', 'w') as f:
    json.dump(splits, f)

print(f'train: {len(train_idx)} | val: {len(val_idx)} | test: {len(test_idx)}')

# Kiểm tra phân phối category
for name, idx in splits.items():
    dist = df_meta.loc[idx, 'category'].value_counts(normalize=True).round(3)
    print(f'  {name}: {dist.to_dict()}')

In [ ]:
print('SANITY CHECK')
print(f'metadata shape     : {df_meta.shape}')
print(f'interactions shape : {df_inter.shape}')
print(f'\nCategory breakdown:\n{df_meta["category"].value_counts()}')
print(f'\nRating distribution:\n{df_inter["rating"].value_counts().sort_index()}')

# NaN check
na = df_meta.isnull().sum()
print(f'\nNaN: {na[na>0].to_dict() or "không có ✓"}')

# Xem thử 4 ảnh đã resize
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
fig.suptitle('Mẫu ảnh sau khi resize 224×224', fontsize=12)
for ax, (_, row) in zip(axes, df_meta.sample(4, random_state=0).iterrows()):
    img = Image.open(row['image_path'])
    ax.imshow(img)
    ax.set_title(f"{row['category']}\n{row['description'][:30]}...", fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sample_images.png', dpi=100)
plt.show()
print('\nTất cả output tại:', OUTPUT_DIR)